# Étape 6.1-6.3 : Feature Engineering

Géo-enrichissement, extraction de features textuelles et encodage.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.geo_enrichment import enrich_geo
from src.feature_engineering import (
    ImputationConfig, QuartierEncoder, build_features,
    add_sdb_indicator, FEATURE_COLS
)

train = pd.read_csv('../data/processed/train_imputed.csv', parse_dates=['date_publication'])
test  = pd.read_csv('../data/processed/test_imputed.csv',  parse_dates=['date_publication'])
print(f"Train : {train.shape}, Test : {test.shape}")

## 6.1 Géo-enrichissement

In [ ]:
train = enrich_geo(train)
test  = enrich_geo(test)

print("Features géo ajoutées:")
geo_cols = ['latitude', 'longitude', 'dist_centre_km', 'dist_aeroport_km', 'dist_plage_km',
            'nb_ecoles_1km', 'nb_mosquees_1km', 'nb_commerces_1km', 'nb_hopitaux_1km', 'nb_total_pois_1km']
print(train[geo_cols].head())
print(f"\nDistance centre (moy): {train['dist_centre_km'].mean():.2f} km")

## 6.2 Feature Engineering

### Imputation (fit sur train uniquement — pas de leakage)

In [ ]:
# Recréer l'imputer (on repart des données imputées basiques + on refait le pipeline complet)
# Note: has_sdb_info déjà créé dans notebook 2
imputer = ImputationConfig()
imputer.fit(train)

# Target encoding sur train uniquement
quartier_enc = QuartierEncoder()
quartier_enc.fit(train, target_col='prix')

print("Target encoding par quartier (prix moyen):")
for q, v in sorted(quartier_enc.target_enc.items(), key=lambda x: -x[1]):
    print(f"  {q}: {v:,.0f} MRU")

In [ ]:
# Build features sur train
train_feat = build_features(train, imputer, quartier_enc, is_train=True)

# Calculer la médiane de taille_rue APRÈS extraction
imputer.fit_taille_rue(train_feat)

# Recalculer avec taille_rue correcte
train_feat = build_features(train, imputer, quartier_enc, is_train=True)
imputer.fit_feature_medians(train_feat)
train_feat = imputer.apply_feature_medians(train_feat)

# Build features sur test (même imputer, pas de leakage)
test_feat = build_features(test, imputer, quartier_enc, is_train=False)

print(f"Train features shape : {train_feat.shape}")
print(f"Test  features shape : {test_feat.shape}")
print(f"\nFeatures extraites ({len(FEATURE_COLS)}) :")
print(FEATURE_COLS)

In [ ]:
# Vérification NaN
print("NaN dans les features (TRAIN):")
nan_counts = train_feat[FEATURE_COLS].isna().sum()
print(nan_counts[nan_counts > 0])

print("\nNaN dans les features (TEST):")
nan_counts_t = test_feat[FEATURE_COLS].isna().sum()
print(nan_counts_t[nan_counts_t > 0])

if nan_counts.sum() == 0 and nan_counts_t.sum() == 0:
    print("Aucun NaN dans les features !")

In [ ]:
# Statistiques descriptives des features
print("Aperçu des features :")
train_feat[FEATURE_COLS].describe().T[['mean', 'std', 'min', 'max']].round(2)

In [ ]:
# Sauvegarder les datasets enrichis
train_feat.to_csv('../data/processed/enriched_data.csv', index=False)
test_feat.to_csv('../data/processed/test_enriched.csv',  index=False)
print("Données enrichies sauvegardées.")
print(f"enriched_data.csv : {train_feat.shape}")